In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
from pyspark.sql.functions import (
    col, when, avg, to_timestamp
)
import os
spark = SparkSession.builder \
    .appName("Weather ETL Pipeline") \
    .config("spark.driver.extraClassPath", "/home/jovyan/work/Grad2/mssql-jdbc-13.2.1.jre8.jar") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
schema = StructType([
    StructField("measurement_id", StringType()),
    StructField("device_id", StringType()),
    StructField("location_id", StringType()),
    StructField("location_name", StringType()),
    StructField("temperature", DoubleType()),
    StructField("humidity", IntegerType()),
    StructField("pressure", IntegerType()),
    StructField("air_quality", IntegerType()),
    StructField("wind_speed", DoubleType()),
    StructField("wind_direction", StringType()),
    StructField("rain_level_mm", DoubleType()),
    StructField("visibility_km", DoubleType()),
    StructField("uv_index", IntegerType()),
    StructField("cloud_coverage_percent", IntegerType()),
    StructField("noise_level", IntegerType()),
    StructField("battery_level", IntegerType()),
    StructField("season", StringType()),
    StructField("day_period", StringType()),
    StructField("timestamp", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("raw_message", StringType())
])
data_path = "/home/jovyan/work/Grad2/S_WEATHER_Batches/*.csv"
df = spark.read.option("header", True).schema(schema).csv(data_path)
df = df.withColumn("timestamp", to_timestamp("timestamp", "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"))
df = df.withColumn(
    "anomaly_flag",
    when((col("temperature") > 40) | (col("temperature") < 10), 1).otherwise(0)
)
df = df.filter((col("temperature") >= -10) & (col("temperature") <= 60))
df = df.withColumn(
    "temperature_status",
    when(col("temperature") > 30, "High")
    .when(col("temperature") < 22, "Low")
    .otherwise("Normal")
)
agg_df = df.groupBy("device_id").agg(
    avg("temperature").alias("avg_temp"),
    avg("humidity").alias("avg_humidity"),
    avg("pressure").alias("avg_pressure"),
    avg("air_quality").alias("avg_air_quality"),
    avg("noise_level").alias("avg_noise_level"),
    avg("battery_level").alias("avg_battery_level"),
)
print("\nDisplaying 10 rows of Aggregated Summary:")
agg_df.show(10)
locations_df = df.select("location_id", "location_name").dropDuplicates()
sensors_df = df.select("device_id", "location_id").dropDuplicates()
measurements_df = df
jdbc_url = "jdbc:sqlserver://host.docker.internal:1433;databaseName=iot_db1;encrypt=false;"
props = {
    "user": "sa",
    "password": "123456789",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
locations_df.write.jdbc(
    url=jdbc_url,
    table="Locations",
    mode="append",
    properties=props
)
sensors_df.write.jdbc(
    url=jdbc_url,
    table="Weather_Sensors",
    mode="append",
    properties=props
)
measurements_df.write.jdbc(
    url=jdbc_url,
    table="Weather_Measurements",
    mode="append",
    properties=props
)
agg_df.write.jdbc(
    url=jdbc_url, 
    table="Batch_iot_summary", 
    mode="append", 
    properties=props
)
print("\n✔ ETL Load Completed Successfully.\n")
spark.stop()


Displaying 10 rows of Aggregated Summary:
+---------+------------------+------------------+------------------+------------------+-----------------+------------------+
|device_id|          avg_temp|      avg_humidity|      avg_pressure|   avg_air_quality|  avg_noise_level| avg_battery_level|
+---------+------------------+------------------+------------------+------------------+-----------------+------------------+
| sensor 5| 36.47684210526315| 54.36842105263158|1024.5263157894738| 53.89473684210526| 70.6842105263158| 48.68421052631579|
| sensor 2|34.897499999999994|              52.7|           1006.55|             55.85|             60.6|              45.8|
| sensor 3|31.124000000000002|             60.05|           1009.15|              45.8|            73.75|              56.8|
| sensor 4|31.242142857142856|              58.0|1016.9285714285714|              48.0|64.64285714285714|              65.0|
| sensor 1| 36.13666666666667|55.333333333333336|1016.8333333333334|46.58333333333